# AES-128 Decryption Notebook
This notebook decrypts data from ciphertext.txt, shows matrix states after every inverse AES step, and verifies plaintext and round relationships.

In [ ]:
from pathlib import Path
import json

from aes_key_expansion import expand_key
from aes_state import bytes_to_state, state_to_bytes
from aes_mix_columns import inv_mix_columns
from aes_sbox import inv_sub_bytes
from aes_shift_rows import inv_shift_rows
from aes_xor import add_round_key
from hash import polynomial_rolling_hash


def matrix_to_hex_rows(state):
    return [" ".join(f"{value:02x}" for value in row) for row in state]


def read_text(path):
    with open(path, "r", encoding="utf-8") as file:
        return file.read()


def read_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


ciphertext_path = Path('ciphertext.txt')
key_path = Path('key.txt')
plaintext_path = Path('plaintext.txt')
enc_trace_path = Path('encryption_trace.json')

if not ciphertext_path.exists() or not key_path.exists() or not plaintext_path.exists() or not enc_trace_path.exists():
    raise FileNotFoundError('Run Encryption notebook first to generate ciphertext.txt, key.txt, plaintext.txt, and encryption_trace.json.')

ciphertext_payload = read_text('ciphertext.txt').strip()
if '|' not in ciphertext_payload:
    raise ValueError('ciphertext.txt must be in format: <ciphertext_hex>|<hash6>')

ciphertext_hex, stored_hash = ciphertext_payload.split('|', 1)
computed_hash = polynomial_rolling_hash(ciphertext_hex, width=6)
if computed_hash != stored_hash:
    raise ValueError(f'Hash mismatch: stored={stored_hash}, computed={computed_hash}')

key_hex = read_text('key.txt').strip()
original_plaintext = read_text('plaintext.txt').strip()
encryption_trace = read_json('encryption_trace.json')

ciphertext_bytes = bytes.fromhex(ciphertext_hex)
if len(key_hex) != 32:
    raise ValueError('key.txt must contain exactly 32 hex characters (16 bytes).')
try:
    key_bytes = bytes.fromhex(key_hex)
except ValueError as error:
    raise ValueError('key.txt must contain valid hexadecimal characters only.') from error


def show_state(label, state):
    print(label)
    for row in matrix_to_hex_rows(state):
        print('  ' + row)
    print()


print('Ciphertext (hex):', ciphertext_hex)
print('Stored hash:', stored_hash)
print('Computed hash:', computed_hash)
print('Hash verified: True')
print('Key (hex):', key_hex)

In [ ]:
# expands the same AES key into round keys for decryption
round_keys = expand_key(key_bytes)

# converts ciphertext bytes into AES 4x4 state matrix
state = bytes_to_state(ciphertext_bytes)

# stores decryption states for round-by-round comparison
decryption_trace = {
    'initial_ciphertext_state': [row[:] for row in state],
    'after_initial_add_round_key': None,
    'rounds': []
}

# initial AddRoundKey for decryption uses last encryption round key
show_state('Initial Ciphertext State', state)
state = add_round_key(state, round_keys[10])
decryption_trace['after_initial_add_round_key'] = [row[:] for row in state]
show_state('After Initial AddRoundKey', state)

In [ ]:
round_number = 1
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

# from round 1 to round 9, inverse AES does 4 steps: InvShiftRows, InvSubBytes, AddRoundKey, InvMixColumns
state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 1 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 1 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 1 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 1 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 2
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 2 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 2 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 2 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 2 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 3
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 3 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 3 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 3 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 3 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 4
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 4 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 4 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 4 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 4 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 5
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 5 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 5 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 5 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 5 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 6
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 6 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 6 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 6 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 6 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 7
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 7 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 7 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 7 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 7 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 8
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 8 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 8 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 8 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 8 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 9
encryption_round = 10 - round_number
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 9 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 9 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 9 - After AddRoundKey', state)

state = inv_mix_columns(state)
round_trace['after_inv_mix_columns'] = [row[:] for row in state]
show_state('Round 9 - After InvMixColumns', state)

decryption_trace['rounds'].append(round_trace)

In [ ]:
round_number = 10
encryption_round = 0
round_trace = {'round': round_number, 'maps_to_encryption_round': encryption_round}

# final decryption round skips InvMixColumns, then converts final state back to plaintext
state = inv_shift_rows(state)
round_trace['after_inv_shift_rows'] = [row[:] for row in state]
show_state('Round 10 - After InvShiftRows', state)

state = inv_sub_bytes(state)
round_trace['after_inv_sub_bytes'] = [row[:] for row in state]
show_state('Round 10 - After InvSubBytes', state)

state = add_round_key(state, round_keys[encryption_round])
round_trace['after_add_round_key'] = [row[:] for row in state]
show_state('Round 10 - After AddRoundKey', state)

decryption_trace['rounds'].append(round_trace)

decrypted_bytes = state_to_bytes(state)
decrypted_text = decrypted_bytes.decode('utf-8')
print('Decrypted text (plaintext):', decrypted_text)

decryption_trace['plaintext_state'] = [row[:] for row in state]

In [ ]:
# cross-check a known AES inverse relation between encryption and decryption traces
enc_r1_after_ark = encryption_trace['rounds'][0]['after_add_round_key']
enc_r9_after_ark = encryption_trace['rounds'][8]['after_add_round_key']
dec_r1_after_inv_sub = decryption_trace['rounds'][0]['after_inv_sub_bytes']
dec_r9_after_inv_sub = decryption_trace['rounds'][8]['after_inv_sub_bytes']

check_1 = enc_r1_after_ark == dec_r9_after_inv_sub
check_2 = enc_r9_after_ark == dec_r1_after_inv_sub

print('Encryption Round 1 (after AddRoundKey) == Decryption Round 9 (after InvSubBytes):', check_1)
print('Encryption Round 9 (after AddRoundKey) == Decryption Round 1 (after InvSubBytes):', check_2)